In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
X, Y = np.meshgrid(np.arange(0, 2 * np.pi, .2), np.arange(0, 2 * np.pi, .2))
U = np.cos(X)
V = np.sin(Y)

Q = plt.quiver(X, Y, U, V, units='width', color='green')
plt.show()

In [ ]:
def generate_hexagonal_seed(point_number, boxsize, lattice_noise=0):
    """
    Create seed points (Voronoi center) on a controlled or "noisy" (with a bit of noise) hexagonal lattice, and applies periodic boundary conditions.
    
    Parameters
    -----
    point_number : int
        Number of seed point to create
    boxsize : float
        Length of the axis for observation box (assumed square for now)
    lattice_noise : float
        Intensity of the noise applied to the lattice seed points

    Returns
    -----
    generated_points : ndarray of ndarray
        Array of seed points on a lattice (of coordinates x,y)
    generated_angles : ndarray
        Array of randomly drawn angles in radians
    generated_polarities : ndarray of double
        Array of polarity vectors (cos theta_i, sin theta_i), initialized with random initial values 
    (in the future maybe): flag
    """
    # FIXME Step should be computed and not hard coded
    cpt = 0
#    generated_points = np.zeros([int(boxsize)**2,2])
    generated_points = np.zeros([point_number,2])
    step = np.sqrt(boxsize**2/point_number)
    print("Step = {}".format(step))
    for line_number,y in enumerate(np.arange(-boxsize/2.0,boxsize/2.0,step)):
        if(line_number % 2 == 0):
            for x in np.arange(-boxsize/2.0,boxsize/2.0,step):
                generated_points[cpt][0] = x + np.random.random() * lattice_noise
                generated_points[cpt][1] = y + np.random.random() * lattice_noise             
                cpt = cpt + 1
        else:
            for x in np.arange( (-boxsize/2.0)+step/2,boxsize/2.0,step):
                generated_points[cpt][0] = x + np.random.random() * lattice_noise 
                generated_points[cpt][1] = y + np.random.random() * lattice_noise               
                cpt = cpt + 1

    return generated_points


In [ ]:
point_number = 100
boxsize = 10
generated_points = generate_hexagonal_seed(point_number, boxsize, lattice_noise=0)
plt.plot(generated_points[:,0],generated_points[:,1], 'ok')
plt.show()
print(len(generated_points))
#print(generated_points)

In [ ]:
def compute_orientational_orders(point_number, iteration_number, velocities):
    """
    Computes the average normalized velocity

    Parameters:
    -----
    # TODO 
    
    Returns:
    -----
    # TODO
    """
        # Break the velocity vector into "iteration_number" pieces of "point_number" elements (with x and y components)
    # We assume that the array is regular with the same number of velocities during each iteration
    velocities_vector = np.reshape(velocities, [iteration_number, point_number,2])

     
    average_absolute_velocity = np.linalg.norm(velocities_vector, axis=2)
    average_absolute_velocity = np.mean(average_absolute_velocity, axis=1)

    orientational_orders = np.sum(velocities_vector, axis=1)
    orientational_orders = np.linalg.norm(orientational_orders, axis=1)
    orientational_orders = orientational_orders / (point_number * average_absolute_velocity[:])

    return orientational_orders

def plot_orientational_orders(orientational_orders, delta_t):
    """

    """
    nb_iter = len(orientational_orders)

    ## Plot the MSD
    # Prepare a time vector to plot the MSD at every time step
    time_vector = np.arange(0, nb_iter*delta_t, delta_t)

    # Fitting
    fig = plt.figure("Evolution of orientational order over time")
    plt.plot(time_vector,  orientational_orders, 'g+')
    plt.title("Evolution of orientational order over time")
    plt.xlabel("Time")
    plt.ylabel(r"$\varphi$")

    return fig



In [ ]:
point_number = 10
iteration_number = 10
velocities = np.ones([iteration_number, point_number, 2])
delta_t = 0.01
#tab = np.arange(point_number)
tab = np.ones(point_number)
tab[0:5] = tab[0:5] * -1
for velocities_iter in velocities:
    velocities_iter[:,0] = velocities_iter[:,0] * tab[:]
    velocities_iter[:,1] = velocities_iter[:,1] * tab[:]
    

print(velocities)
orientational_orders = compute_orientational_orders(point_number, iteration_number, velocities)
fig_orientational_order = plot_orientational_orders(orientational_orders, delta_t)

In [ ]:
def gaussian_white_noise_process(mean, std, sample_number):
    """
    Draws random samples from a gaussian distribution

    Parameters:
    mean : double
        Mean of the Gaussian Distribution
    std : double 
        Standard deviation of the white noise process
    sample_number : int
        Number of samples to generate
    ----- 
    Returns:
    white_noise_samples : numpy array
        Array of random white noise samples (double)
    """

    white_noise_samples = np.random.normal(mean, std, sample_number)
    return white_noise_samples

def compute_every_random_rotational_diffusion_euler(angles, angular_noise,delta_t):
    """
    Compute the values at next timestep for every angle assuming random rotational diffusion (Giavazzi (3)) 
    and using Euler's method

    Parameters:
    angles: numpy array
        Theta values in radian for the polarity vector
    angular_noise : double
        D_r term, determines the memory of stochastic noise in the system
    delta_t : double
        Time integration step
    -----    
    Returns:
    angles_next : numpy array
        Theta values in radian for the next timestep
    """
    
    random_process_samples = gaussian_white_noise_process(0, angular_noise*2, len(angles)) 
    angles_next = delta_t * random_process_samples + angles 

    # Modulo 2 pi to normalize
    angles_next = np.mod(angles_next,2*np.pi)

    return angles_next 


In [ ]:
white_noise_samples = gaussian_white_noise_process(0, 2, 100)
plt.hist(white_noise_samples)
plt.show()

In [ ]:
np.random.seed(111)
angles = np.ones(100) * 3.10
ang_noise = 1
delta_t = 1
angles_next = compute_every_random_rotational_diffusion_euler(angles, ang_noise, delta_t)
print(angles_next)
print('-'*10)
print(angles_next-angles)

In [ ]:
### Polar histogram
point_number = 10
radii = 10 * np.random.rand(point_number)
width = (2*np.pi) / point_number

# Initialize the vector
velocity_angles = np.ones(point_number) 


velocity_angles[0:5] = velocity_angles[0:5] * np.pi/2
velocity_angles[5:10] = velocity_angles[5:10] * 0
velocity_angles[0] = np.pi

fig = plt.figure("Classic histogram")
plt.hist(velocity_angles, bins='auto')
plt.title("Distribution of velocity angles")
plt.show()


fig = plt.figure("Polar histogram")
ax = plt.subplot(111, projection='polar')
plt.hist(velocity_angles, bins='auto')
#bars = ax.bar(velocity_angles, radii, width=width)
plt.title("Distribution of velocity angles")

plt.show()
print(velocity_angles)

In [ ]:
#######################################################################